# Lead-conversion prediction

**Goal:** an online-learning startup gets more leads than its sales team can call. I want to rank leads by how likely they are to become paying customers, and find what drives conversion so sales time and marketing budget go where they pay off.

The reusable code lives in `src/leadscore/`; this notebook walks through the analysis and the reasoning.

**Data:** 4,612 leads supplied as course material. It is not public and is not committed to this repo; `data/README.md` describes the schema. Set `LEADS_SYNTHETIC=1` to run the notebook on generated data.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from leadscore import config
from leadscore.analysis import conversion_by, conversion_by_channel
from leadscore.data import load_leads
from leadscore.evaluate import confusion, feature_importance, score, tree_rules
from leadscore.features import encode, split
from leadscore.synthetic import make_leads
from leadscore.train import default_forest, default_tree, tuned_forest, tuned_tree

sns.set_theme(style="whitegrid")
print("Seed:", config.SEED)

## Load and check the data

In [ ]:
DATA_PATH = Path(os.environ.get("LEADS_DATA", "../data/raw/leads.csv"))
SYNTHETIC = os.environ.get("LEADS_SYNTHETIC") == "1"

leads = make_leads() if SYNTHETIC else load_leads(DATA_PATH)
print(leads.shape)
print("Duplicate rows:", leads.duplicated().sum())
print("Missing values:", int(leads.isna().sum().sum()))
leads.describe().T

In the original run: 4,612 leads, no missing values, no duplicates, every ID unique (so the ID carries no information and is dropped before modeling). About 30% of leads converted. Ages run from 18 to 63. Time on the website (seconds) is heavily skewed, from 0 to about 42 minutes.

In [ ]:
leads[config.TARGET].value_counts(normalize=True).rename("share")

## What drives conversion? A first look

Conversion rate by occupation, first interaction, last activity and profile completion.

In [ ]:
segments = ["current_occupation", "first_interaction", "last_activity", "profile_completed"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, segments, strict=True):
    table = conversion_by(leads, col)
    table["rate"].plot(kind="bar", ax=ax, color="#3274A1")
    ax.set(title=col.replace("_", " ").capitalize(), ylabel="Conversion rate", xlabel="")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

Original run:

- **Occupation:** professionals convert most often, then unemployed leads, then students.
- **First interaction:** leads whose first touch was the website convert far more often than mobile-app leads.
- **Last activity:** email and website activity outperform phone follow-up.
- **Profile completion:** high completion converts best.

## Which marketing channel converts best?

A lead can have seen several channels, so I compute the conversion rate among the leads who saw each one.

In [ ]:
channels = conversion_by_channel(leads)
channels.assign(rate=channels["rate"].map("{:.1%}".format))

Referrals converted at **67.7%** (63 of 93 referred leads) in the original run; every other channel was close to 30% (magazine 32.2%, newspaper 32.0%, digital ads 31.9%, education channels 27.9%). Referrals are by far the strongest channel but also the smallest.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, config.NUMERIC_COLUMNS, strict=True):
    sns.boxplot(data=leads, x=config.TARGET, y=col, ax=ax, showfliers=False)
    ax.set(title=col.replace("_", " ").capitalize(), xlabel="Converted")
plt.tight_layout()
plt.show()
leads[config.NUMERIC_COLUMNS + [config.TARGET]].corr()[config.TARGET].round(2)

Time on the website is the only numeric feature with a clear relationship to conversion; age, visits and pages per visit are weakly related.

## Prepare the data

One-hot encode the categorical fields (dropping the first level of each), then split 70/30, stratified on conversion so both sides keep the same conversion rate.

In [ ]:
x, y = encode(leads)
x_train, x_test, y_train, y_test = split(x, y)
print("Features:", x.shape[1])
print("Train:", x_train.shape, "Test:", x_test.shape)
print(f"Conversion rate train {y_train.mean():.3f} / test {y_test.mean():.3f}")

## What to optimize

A missed buyer is lost revenue; an extra call to a lead who doesn't buy costs a few minutes. So I score models on **recall for the converted class**: the share of real buyers the model finds. Accuracy would also look good on a model that ignores the 30% minority, so I report precision and F1 alongside it.

In [ ]:
results = {}


def report(name, model):
    test = score(y_test, model.predict(x_test))
    train_acc = score(y_train, model.predict(x_train)).accuracy
    results[name] = {**test.rounded(), "train_accuracy": round(train_acc, 2)}
    display(confusion(y_test, model.predict(x_test)))
    return results[name]

## Model 1: decision tree, default settings

In [ ]:
tree = default_tree(x_train, y_train)
report("Decision tree, default", tree)

Training accuracy is 1.00 and test accuracy much lower: the unpruned tree memorizes the training data. It needs pruning.

## Model 2: decision tree, pruned by grid search (5-fold CV, scored on recall)

In [ ]:
pruned, tree_params = tuned_tree(x_train, y_train)
print(tree_params)
report("Decision tree, pruned", pruned)

In [ ]:
print(tree_rules(pruned, list(x.columns), max_depth=3))

The pruned tree reads like a sales playbook. In the original run it first split on whether the first interaction was the website, then on **time on the website at about 414 to 420 seconds (roughly 7 minutes)**:

- Website-first leads with 7+ minutes on the site are predicted to convert. With less time, profile completion decides.
- Mobile-app-first leads are only predicted to convert if they later spend 7+ minutes on the site and their last activity was on the website.

## Model 3: random forest, default settings

In [ ]:
forest = default_forest(x_train, y_train)
report("Random forest, default", forest)

Best accuracy so far, but it also scores 1.00 on the training data, and it finds fewer of the buyers than the pruned tree.

## Model 4: random forest, tuned by grid search (3-fold CV, scored on recall)

In [ ]:
tuned, forest_params = tuned_forest(x_train, y_train)
print(forest_params)
report("Random forest, tuned", tuned)

In [ ]:
importance = feature_importance(tuned, list(x.columns)).head(15)
importance.sort_values().plot(kind="barh", figsize=(8, 6), color="#3274A1")
plt.title("Tuned random forest: top 15 feature importances")
plt.tight_layout()
plt.show()

## Compare

In [ ]:
pd.DataFrame(results).T

Original run, test set of 1,384 leads (converted class):

| Model | Accuracy | Recall | Precision | F1 |
|---|---|---|---|---|
| Decision tree, default | 0.79 | 0.63 | 0.66 | 0.64 |
| Decision tree, pruned (depth 3) | 0.78 | 0.89 | 0.58 | 0.70 |
| Random forest, default | 0.86 | 0.72 | 0.79 | 0.76 |
| **Random forest, tuned** | **0.83** | **0.85** | **0.67** | **0.75** |

**My choice: the tuned random forest.** Against the pruned tree it gives up 4 points of recall for 9 points of precision; against the most accurate model it finds 13 points more buyers. Its train/test accuracy gap is 2 points (0.85 vs 0.83), against 14 for the default forest. I keep the pruned tree for explaining the "why" to sales.

**Caveat:** I compared all four models on the same test set, so the winner's scores are somewhat optimistic. Next time I'd choose on cross-validated scores and keep the test set for one final number.

## What I'd tell the business

- **Sales:** call website-first leads with 7+ minutes on the site first. The tuned model's list catches about 85% of eventual buyers, and about two in three flagged leads convert.
- **Marketing:** grow the referral program; it converts best but is the smallest channel. Make profile completion easier and invest in website content that keeps people reading.
- **Product:** find out why mobile-app leads convert less.

## What I'd do next

- Tune the probability threshold with sales against real call costs and deal values instead of using 0.5.
- Try gradient-boosted trees and calibrate the probabilities.
- Check time on site for leakage: if long sessions happen *after* a sales conversation, the model partly sees the outcome.
- Run a controlled trial of the ranked list against the current process.